Ingest GlobalMart sales data from csv into bronze table

In [0]:
# import libraries 
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re


In [0]:
# define file path
file_path = "/Volumes/global_mart_retail/bronze/raw_data/Sample - Superstore.csv"

In [0]:
df_raw = spark.read.format("csv").option("header","true").option("inferschema","true").load(file_path)

# the columns all have spaces in between them, this is not convinent for storage in databricks
# the columns will need to be sanitised
def sanitize_column(name):
    return re.sub(r"[ ,;{}()\n\t=\-]", "_", name)

for column_name in df_raw.columns:
    df_raw = df_raw.withColumnRenamed(column_name, sanitize_column(column_name))


# Pipeline metadata - added by our ingestion layer for tracking and debugging
df_raw = (
            df_raw.withColumn("ingest_timestamp" , current_timestamp())
                   .withColumn("source", lit("initial_load"))
                 
          )
                

# writing to bronze table
df_raw.write.mode("overwrite").saveAsTable("global_mart_retail.bronze.orders")

print("Data written to bronze table successfully")